# 18 — CTD Rewarded-Soup-Inspired SFT Interpolation

This is a **Rewarded-Soups-inspired** follow-up, not an exact reproduction of Rewarded Soups: the endpoints are trained with SFT distributions rather than RL reward optimization. The question is whether interpolation between a selection-specialized model and a sufficiency-specialized model traces a useful empirical trade-off curve.

We reuse the Experiment 15 Robust SFT adapter as the **selection endpoint** and train a new **sufficiency endpoint** from the same Qwen2.5-0.5B base initialization. For each split and seed, we interpolate the two LoRA deltas with PEFT's `cat` weighted-adapter construction, which represents the weighted sum of the two LoRA updates exactly.

Primary plot: x = Distractor-5 accuracy (evidence selection), y = Hard no-path accuracy (evidence sufficiency), with interpolation coefficient $\lambda\in[0,1]$. We report the empirical non-dominated points rather than claiming the true Pareto frontier.


In [ ]:
!pip -q install -U transformers datasets trl peft accelerate sentencepiece requests

import os,re,gc,json,random,gzip
from pathlib import Path
import numpy as np, pandas as pd, torch, requests
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer

MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
SEEDS=[1,2,3]
SPLITS=['ChemicalID','GeneID','DiseaseID']
N_TRAIN=1500
N_EVAL=100
SFT_STEPS=80
LR=2e-4
LAMBDAS=[round(x,1) for x in np.linspace(0,1,11)]

ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
DATA_DIR=ROOT/'ctd_data'; DATA_DIR.mkdir(parents=True,exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_ROOT=Path('/content/drive/MyDrive/llm-tuning-playground')
except Exception:
    DRIVE_ROOT=ROOT/'llm-tuning-playground'
ROBUST_DIR=DRIVE_ROOT/'results/15/adapters'
RESULT_DIR=DRIVE_ROOT/'results/18'
SUFF_DIR=RESULT_DIR/'sufficiency_adapters'
RESULT_DIR.mkdir(parents=True,exist_ok=True); SUFF_DIR.mkdir(parents=True,exist_ok=True)
RESULT_CSV=RESULT_DIR/'18_soup_results.csv'
SUMMARY_CSV=RESULT_DIR/'18_soup_summary.csv'
CONFIG_JSON=RESULT_DIR/'18_soup_config.json'

config=dict(model=MODEL_NAME,seeds=SEEDS,splits=SPLITS,n_train=N_TRAIN,n_eval=N_EVAL,sft_steps=SFT_STEPS,learning_rate=LR,lambdas=LAMBDAS,selection_endpoint='Experiment 15 robust adapter',sufficiency_endpoint='50/50 distractor-positive and no-path SFT from base',interpolation='exact weighted sum of LoRA deltas via PEFT cat adapter',scope='Rewarded-Soups-inspired SFT interpolation; not exact RL Rewarded Soups')
CONFIG_JSON.write_text(json.dumps(config,indent=2),encoding='utf-8')
print('CUDA:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Results:',RESULT_DIR)


In [ ]:
# CTD acquisition and parser (same benchmark family as Experiments 15--17)
CHEM_NAME='CTD_chem_gene_ixns.tsv.gz'
GD_NAMES=['CTD_curated_genes_diseases.tsv.gz','CTD_genes_diseases.tsv.gz']

def valid_gzip(path,min_bytes=10000):
    path=Path(path)
    if not path.exists() or path.stat().st_size<min_bytes:return False
    try:
        with open(path,'rb') as f:
            if f.read(2)!=b'\x1f\x8b':return False
        with gzip.open(path,'rb') as f:f.read(256)
        return True
    except Exception:return False
def find_local(name):
    for p in [Path.cwd()/name,ROOT/name,DATA_DIR/name,Path('/content/drive/MyDrive')/name,Path('/content/drive/MyDrive/ctd')/name,Path('/content/drive/MyDrive/data')/name]:
        if valid_gzip(p):print('Found:',p);return p
    return None
def download_ctd(name):
    dest=DATA_DIR/name
    for url in [f'https://ctdbase.org/reports/{name}',f'https://ctdbase.org/downloads/{name}',f'http://ctdbase.org/reports/{name}']:
        try:
            print('Trying:',url)
            with requests.get(url,stream=True,timeout=(20,300),allow_redirects=True,headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(dest,'wb') as f:
                    for ch in r.iter_content(1024*1024):
                        if ch:f.write(ch)
            if valid_gzip(dest):return dest
        except Exception as e:print(' failed:',type(e).__name__,str(e)[:120])
        dest.unlink(missing_ok=True)
    return None
def ensure_ctd(names):
    if isinstance(names,str):names=[names]
    for n in names:
        p=find_local(n)
        if p:return p
    for n in names:
        p=download_ctd(n)
        if p:return p
    raise FileNotFoundError(names)
def read_ctd(path,expected_any):
    header=None
    with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            if not line.startswith('#'):break
            s=line.lstrip('#').strip()
            if '\t' in s:
                cols=[x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected_any):header=cols
    if header is None:raise ValueError(path)
    return pd.read_csv(path,sep='\t',comment='#',compression='gzip',dtype=str,low_memory=False,header=None,names=header)
def pick(df,names):
    for n in names:
        if n in df.columns:return n
    raise KeyError(names)

cg=read_ctd(ensure_ctd(CHEM_NAME),['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=read_ctd(ensure_ctd(GD_NAMES),['GeneSymbol','GeneID','DiseaseName','DiseaseID'])
c_name=pick(cg,['ChemicalName']);c_id=pick(cg,['ChemicalID']);g_sym1=pick(cg,['GeneSymbol']);g_id1=pick(cg,['GeneID'])
g_sym2=pick(gd,['GeneSymbol']);g_id2=pick(gd,['GeneID']);d_name=pick(gd,['DiseaseName']);d_id=pick(gd,['DiseaseID'])
cg2=cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates();gd2=gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns=['ChemicalName','ChemicalID','GeneSymbol','GeneID'];gd2.columns=['GeneSymbol','GeneID','DiseaseName','DiseaseID']
paths=cg2.merge(gd2,on=['GeneSymbol','GeneID'],how='inner').drop_duplicates()
paths=paths[(paths.ChemicalName.str.len()<100)&(paths.DiseaseName.str.len()<120)].reset_index(drop=True)
edge_pool=gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)
assert len(paths)>3000
print('Two-hop paths:',len(paths))


In [ ]:
# Benchmark and scoring helpers
def render_prompt(row,edges):
    lines=[f'- {g} -> {d}' for g,d in edges]
    return ('Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'+f'Chemical: {row.ChemicalName}\nGene: {row.GeneSymbol}\nEvidence:\n'+'\n'.join(lines))
def positive_edges(row,k,rng):
    edges=[(str(row.GeneSymbol),str(row.DiseaseName))]
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    if k:
        sub=pool.sample(n=k,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges
def no_path_edges(row,k,rng,lexical=False):
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)].copy();edges=[]
    if lexical:
        sym=str(row.GeneSymbol);near=pool[pool.GeneSymbol.astype(str).str.startswith(sym[:max(1,min(2,len(sym)))])]
        if len(near):
            x=near.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0];edges.append((str(x.GeneSymbol),str(x.DiseaseName)));pool=pool[pool.GeneSymbol!=x.GeneSymbol]
    need=k-len(edges)
    if need>0:
        sub=pool.sample(need,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges
def counterfactual_edges(row,rng):
    c=edge_pool[(edge_pool.GeneSymbol==row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    cf=str(c.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName) if len(c) else str(edge_pool[edge_pool.DiseaseName!=row.DiseaseName].sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    return [(str(row.GeneSymbol),cf)],cf
def answer_text(row):return f'Disease: {row.DiseaseName}. Reasoning: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'
def make_split(df,col,seed):
    r=np.random.default_rng(seed);ents=df[col].dropna().unique().copy();r.shuffle(ents);cut=max(1,int(.8*len(ents)))
    tr_e,te_e=set(ents[:cut]),set(ents[cut:]);trp=df[df[col].isin(tr_e)];tep=df[df[col].isin(te_e)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
    tr=trp.sample(N_TRAIN,random_state=seed).reset_index(drop=True);te=tep.sample(N_EVAL,random_state=1000+seed).reset_index(drop=True)
    assert set(tr[col]).isdisjoint(set(te[col]));return tr,te
def item(row,edges,target,typ):return {'target_disease':None if target is None else str(target),'prompt':render_prompt(row,edges),'answer_type':typ}
def make_eval_sets(df,seed):
    rng=random.Random(20000+seed);out={k:[] for k in ['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']}
    for _,row in df.iterrows():
        out['clean'].append(item(row,positive_edges(row,0,rng),row.DiseaseName,'positive'))
        out['distractor_5'].append(item(row,positive_edges(row,5,rng),row.DiseaseName,'positive'))
        out['hard_no_path'].append(item(row,no_path_edges(row,5,rng,False),None,'no_path'))
        out['lexical_no_path'].append(item(row,no_path_edges(row,5,rng,True),None,'no_path'))
        e,cf=counterfactual_edges(row,rng);out['counterfactual'].append(item(row,e,cf,'positive'))
    return out
def norm(s):return re.sub(r'\s+',' ',str(s).strip().lower())
def score_one(x,p):
    p=norm(p)
    return ('no supported path' in p) if x['answer_type']=='no_path' else (norm(x['target_disease']) in p and 'no supported path' not in p)
def score_set(items,preds):return float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))


In [ ]:
# Train the sufficiency endpoint from the same base initialization.
# 50% distractor-positive, 50% no-path; unlike the selection endpoint, it receives direct sufficiency supervision.
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True)
tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token

def make_sufficiency_sft(df,seed):
    rng=random.Random(40000+seed);rows=[]
    for _,row in df.iterrows():
        if rng.random()<0.5:
            edges=no_path_edges(row,rng.choice([3,5,10]),rng,rng.random()<0.5);target='No supported path.'
        else:
            edges=positive_edges(row,rng.choice([1,3,5,10]),rng);target=answer_text(row)
        rows.append({'text':render_prompt(row,edges)+'\nAnswer: '+target})
    return Dataset.from_list(rows)

def train_sufficiency_adapter(train_df,split,seed):
    out=SUFF_DIR/f'sufficiency_{split}_{seed}'
    if (out/'adapter_config.json').exists():
        print('Reuse:',out);return out
    set_seed(seed)
    model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map='auto')
    lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type=TaskType.CAUSAL_LM,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model=get_peft_model(model,lora)
    ds=make_sufficiency_sft(train_df,seed)
    args=SFTConfig(output_dir=str(RESULT_DIR/'tmp_sft'),max_steps=SFT_STEPS,per_device_train_batch_size=2,gradient_accumulation_steps=8,learning_rate=LR,logging_steps=20,save_strategy='no',report_to='none',fp16=torch.cuda.is_available(),bf16=False,max_length=512,dataset_text_field='text')
    trainer=SFTTrainer(model=model,args=args,train_dataset=ds,processing_class=tokenizer)
    trainer.train();model.save_pretrained(out);tokenizer.save_pretrained(out)
    del trainer,model;gc.collect();torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print('Saved:',out);return out


In [ ]:
# Generation and exact LoRA-delta interpolation.
def generate(model,items,batch_size=16):
    model.eval();outs=[]
    for i in range(0,len(items),batch_size):
        prompts=[x['prompt']+'\nAnswer:' for x in items[i:i+batch_size]]
        enc=tokenizer(prompts,return_tensors='pt',padding=True,truncation=True,max_length=512).to(model.device)
        with torch.no_grad():gen=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        for j,g in enumerate(gen):outs.append(tokenizer.decode(g[enc['input_ids'].shape[1]:],skip_special_tokens=True))
    return outs

def load_two_endpoint_model(selection_path,suff_path):
    base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map='auto')
    model=PeftModel.from_pretrained(base,str(selection_path),adapter_name='selection',is_trainable=False)
    model.load_adapter(str(suff_path),adapter_name='sufficiency',is_trainable=False)
    return model

def set_lambda_adapter(model,lam):
    # Endpoints use the original adapters. Interior points use PEFT `cat`, which concatenates LoRA factors so the resulting update equals (1-lambda)*DeltaW_selection + lambda*DeltaW_sufficiency.
    if lam==0:return 'selection'
    if lam==1:return 'sufficiency'
    name=f'soup_{lam:.1f}'.replace('.','p')
    model.add_weighted_adapter(adapters=['selection','sufficiency'],weights=[1-lam,lam],adapter_name=name,combination_type='cat')
    return name


In [ ]:
# Main loop with incremental checkpointing.
existing=pd.read_csv(RESULT_CSV) if RESULT_CSV.exists() else pd.DataFrame()
rows=[] if existing.empty else existing.to_dict('records')
done={(r['split'],int(r['seed']),float(r['lambda']),r['condition']) for r in rows}

for split in SPLITS:
  for seed in SEEDS:
    print('\n===',split,'seed',seed,'===')
    train_df,test_df=make_split(paths,split,seed);eval_sets=make_eval_sets(test_df,seed)
    selection_path=ROBUST_DIR/f'robust_{split}_{seed}'
    if not (selection_path/'adapter_config.json').exists():raise FileNotFoundError(f'Missing Experiment 15 robust adapter: {selection_path}')
    suff_path=train_sufficiency_adapter(train_df,split,seed)
    model=load_two_endpoint_model(selection_path,suff_path)
    for lam in LAMBDAS:
        if all((split,seed,float(lam),c) in done for c in eval_sets):continue
        adapter=set_lambda_adapter(model,lam);model.set_adapter(adapter)
        for cond,items in eval_sets.items():
            key=(split,seed,float(lam),cond)
            if key in done:continue
            preds=generate(model,items);acc=score_set(items,preds)
            rows.append({'split':split,'seed':seed,'lambda':lam,'condition':cond,'accuracy':acc,'adapter':adapter})
            done.add(key);pd.DataFrame(rows).to_csv(RESULT_CSV,index=False)
            print(f'lambda={lam:.1f} {cond}: {acc:.3f}')
        if adapter.startswith('soup_'):
            model.delete_adapter(adapter)
    del model;gc.collect();torch.cuda.empty_cache() if torch.cuda.is_available() else None

res=pd.DataFrame(rows)
summary=res.groupby(['lambda','condition']).accuracy.agg(['mean','std','count']).reset_index()
summary.to_csv(SUMMARY_CSV,index=False)
display(summary)


In [ ]:
# Empirical Pareto analysis on the two primary axes.
means=res.groupby(['lambda','condition']).accuracy.mean().unstack('condition')
pts=means[['distractor_5','hard_no_path']].dropna().copy()
def nondominated(df):
    keep=[]
    for i,row in df.iterrows():
        dominated=((df.distractor_5>=row.distractor_5)&(df.hard_no_path>=row.hard_no_path)&((df.distractor_5>row.distractor_5)|(df.hard_no_path>row.hard_no_path))).any()
        if not dominated:keep.append(i)
    return df.loc[keep].sort_values('distractor_5')
front=nondominated(pts)
print('Empirical non-dominated operating points (D5 vs Hard NP):')
display(front)
front.reset_index().to_csv(RESULT_DIR/'18_empirical_pareto_points.csv',index=False)

# Optional quick visualization.
import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
plt.plot(pts.distractor_5,pts.hard_no_path,'o-')
for lam,row in pts.iterrows():plt.annotate(f'{lam:.1f}',(row.distractor_5,row.hard_no_path))
plt.xlabel('Distractor-5 accuracy (selection)');plt.ylabel('Hard no-path accuracy (sufficiency)');plt.title('Experiment 18: interpolation trade-off')
plt.grid(alpha=.25);plt.show()


## Interpretation guardrails

- Call this an **empirical interpolation trade-off curve** or **empirical non-dominated set**, not the true Pareto frontier.
- This notebook is **Rewarded-Soups-inspired**, because the endpoint models are SFT-trained rather than reward-optimized with RL.
- A useful result would be a smooth trajectory with one or more interior points improving the observed selection--sufficiency trade-off relative to the endpoints.
- If interpolation is erratic or dominated, that is also informative: it suggests that the two behaviors are not linearly connected in this parameterization/training setup.
